# 02 · Train and submit

A LightGBM baseline, evaluated without fooling yourself, then submitted to the **open round**.
About ten minutes.

Coming from [`01_explore_the_data.ipynb`](01_explore_the_data.ipynb). The submission section
spends a real upload only when you flip `SUBMIT = True`. Your allowance is **account-wide, not
per round**, so spend it deliberately.

In [ ]:
%pip install --quiet "everestapi>=0.3.32" lightgbm pandas pyarrow numpy scipy matplotlib cloudpickle

In [ ]:
import os
from everestapi import EverestAPI

# Read every credential from the environment, so nothing secret is written into the notebook.
base_url = os.environ.get("EIQ_BASE_URL", "https://app.everesteer.ai")
api_key = os.environ.get("EIQ_API_KEY") or os.environ.get("EVEREST_API_KEY")

# Fail fast with a clear message instead of a cryptic 401/403 deeper in the notebook.
if not api_key:
    raise RuntimeError("Set EIQ_API_KEY (onboarding -> Copy setup command).")

# Only the gated STAGING mirror needs a Cloudflare Access service token; the public
# site does not. The SDK picks the CF_ACCESS_* env vars up automatically when set.
if "staging" in base_url and not (
    os.environ.get("CF_ACCESS_CLIENT_ID") and os.environ.get("CF_ACCESS_CLIENT_SECRET")
):
    raise RuntimeError(
        "Staging is behind Cloudflare Access - set CF_ACCESS_CLIENT_ID / "
        "CF_ACCESS_CLIENT_SECRET, or point EIQ_BASE_URL at the public site."
    )

client = EverestAPI(api_key=api_key, base_url=base_url)
client.health()  # returns {'status': 'ok', ...} on a working, authenticated connection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# One small palette across every plot in these notebooks, so charts read as one system.
NAVY, TEAL, CORAL, GREY = "#09142F", "#007B63", "#EC9A5F", "#9AA3B2"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GREY, "axes.grid": True, "grid.color": "#E6E9EF",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
})

In [ ]:
import pandas as pd

# verbose=True returns ONLY feature_sets + targets. Every scalar fact
# (primary_target, feature_encoding, ...) is on the COMPACT response, so
# take membership from one call and the graded column from the other.
schema = client.get_dataset_schema()
schema_verbose = client.get_dataset_schema(verbose=True)
feature_sets = schema_verbose["feature_sets"]
target_cols = list(schema["targets"])
# The graded column, straight from the schema - it differs between datasets and
# is NOT necessarily the first entry in `targets`.
PRIMARY_TARGET = schema["primary_target"]

# download_dataset writes a parquet locally and returns its path; the version auto-resolves.
# `train` is the labeled split and the largest - everything below is built from it.
train = pd.read_parquet(client.download_dataset(universe="futures", split="train"))

EXPED = "exped"   # each exped = one trading day (an "expedition")

# Work with the SMALLEST published feature set (fewer features = faster, less
# memory); scale up later for more signal. Which sets exist is a dataset fact.
FEATURE_SET_NAME = min(feature_sets, key=lambda n: len(feature_sets[n]))
feature_set = feature_sets[FEATURE_SET_NAME]

def exped_num(e):
    # Expeds are ordered strings like "exped_8978"; pull the integer so we can order them in time.
    return int(str(e).split("_")[-1])

# Downsample to every 4th exped purely for speed in this walkthrough; use all of it for real models.
keep = sorted(train[EXPED].unique(), key=exped_num)[::4]
train = train[train[EXPED].isin(keep)]

print(f"train {train.shape}")
print(f"features used: {len(feature_set)} from the {FEATURE_SET_NAME!r} set")

## 1. Carve an honest holdout, before you train

You need somewhere to measure yourself that the model has never seen. With a hackathon key that
place is **not** the `validation` split: its targets are blanked, and it is scored server-side.
You cannot compute CORR on it locally. Build the holdout out of the labeled `train` split instead.

Two rules make it honest:

- **Hold out the *end*, not a random sample.** Random rows leak across time; the tail is the
  closest thing to "the future" you have locally.
- **Embargo the boundary.** The target is a FORWARD return, so the last fitting expeds
  already encode what happens over the first holdout expeds. One exped is one trading day,
  so discarding at least as many expeds as the target looks ahead guarantees zero overlap.
  **The horizon is a dataset fact and is not encoded in the target name on every dataset**
  - when in doubt, embargo generously. (This walkthrough kept only every 4th exped for
  speed, so 20 *kept* expeds is a wider gap than strictly needed,
  erring wide costs nothing. On the full split it is exactly the 20-day horizon.)

Skipping the embargo is the most common way people fool themselves with a good-looking score.
(The same trap applies to hosted training: `train(...)` fits the **whole** split, so a "holdout"
carved from its artifacts afterwards is in-sample.)

In [ ]:
HOLDOUT_EXPEDS = 100   # the tail of train, kept aside
EMBARGO = 20           # expeds discarded between fit and holdout

ordered = sorted(train[EXPED].unique(), key=exped_num)
holdout_expeds = set(ordered[-HOLDOUT_EXPEDS:])
fit_expeds = set(ordered[: -(HOLDOUT_EXPEDS + EMBARGO)])

fit = train[train[EXPED].isin(fit_expeds)]
holdout = train[train[EXPED].isin(holdout_expeds)].copy()

print(f"fit     {len(fit):>8,} rows over {len(fit_expeds):>4} expeds")
print(f"embargo {'':>8}  {EMBARGO:>4} expeds discarded")
print(f"holdout {len(holdout):>8,} rows over {len(holdout_expeds):>4} expeds")
assert not (fit_expeds & holdout_expeds), "fit and holdout must not share expeds"

## 2. Train

Any model works. On noisy, low-signal tabular data like this the usual families are:

- **Tree-based**, the default. They handle binned integer features and missing values
  gracefully and capture interactions without scaling. *LightGBM* / *XGBoost* (fast, strong),
  *Random Forest* (robust, low-tuning), *CatBoost* (first-class categorical handling, which
  suits binned features).
- **Linear**, *Ridge* / *Lasso* / *ElasticNet*. Fast, low-variance, rarely overfit; miss
  interactions but excellent for ensembling and as a neutralization base.
- **Neural nets**, MLPs and tabular nets. Rich interactions, but need careful regularisation
  here; most often used inside an ensemble rather than alone.

Everesteer's hosted `train` tool covers `lightgbm`, `xgboost`, `ridge`, `mlp` and
`random_forest` as presets, plus `model="custom"` for your own code (CatBoost lives there,
it is importable in the custom sandbox but has no preset). See
[`starter_hosted.py`](../starter_hosted.py).

Below: LightGBM locally. Note `features_matrix` maps the `-1` sentinel to `NaN` so LightGBM
treats it as missing rather than as a real bin.

In [ ]:
import lightgbm as lgb

def features_matrix(df, cols):
    # Cast to float and turn the -1 missing sentinel into NaN, so LightGBM treats it as
    # "missing" (it handles NaN natively) instead of as a real bin between 0 and 4.
    X = df[cols].astype("float32")
    return X.where(X >= 0)

# Lightly-regularized starting params (not tuned). The two commented knobs matter most
# on this noisy, low-signal data.
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=6,
    num_leaves=64,
    colsample_bytree=0.1,    # few features per tree -> more diverse trees, less overfit
    min_child_samples=500,   # large leaves -> smoother predictions, less overfit
    random_state=42,
    verbose=-1,
)
model.fit(features_matrix(fit, feature_set), fit[PRIMARY_TARGET])
print("Trained on", f"{len(fit):,}", "rows.")

## 3. Evaluate on the holdout

### Per-exped CORR, Sharpe and drawdown

CORR is the rank correlation between your predictions and the target, computed **within each
exped** and then summarised across expeds.

Sharpe (mean / std of per-exped CORR) and max drawdown are **display-only diagnostics**, they
tell you whether an edge is consistent, but they do not affect your rank. What ranks you is the
round score: CORR20, AIMC and NCORR blended.

In [ ]:
from scipy.stats import spearmanr

holdout["prediction"] = model.predict(features_matrix(holdout, feature_set))

def per_exped_corr(df):
    # Spearman (rank) correlation between prediction and target, one value per exped.
    out = {}
    for e, g in df.groupby(EXPED):
        rho, _ = spearmanr(g["prediction"], g[PRIMARY_TARGET])
        if np.isfinite(rho):
            out[e] = rho
    # Order by exped number so the cumulative curve and drawdown are chronological.
    s = pd.Series(out)
    return s.reindex(sorted(s.index, key=exped_num))

corr = per_exped_corr(holdout.dropna(subset=[PRIMARY_TARGET]))
cum = corr.cumsum()
peak = cum.expanding().max()
drawdown = peak - cum

print(f"Mean CORR {corr.mean():.4f} | Std {corr.std():.4f} | "
      f"Sharpe {corr.mean() / corr.std():.2f} | % positive {(corr > 0).mean() * 100:.0f}%")
print(f"Max drawdown {drawdown.max():.4f}")

In [ ]:
# Cumulative CORR with the drawdown from the running peak shaded underneath.
x = range(len(cum))
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(9, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
)

ax1.plot(x, peak.values, color=GREY, lw=0.8, ls="--", label="running peak")
ax1.plot(x, cum.values, color=NAVY, lw=1.4, label="cumulative CORR")
ax1.fill_between(x, cum.values, peak.values, color=CORAL, alpha=0.25, label="drawdown")
ax1.axhline(0, color=GREY, lw=0.8)
ax1.set(title="Cumulative holdout CORR (embargoed tail of train)", ylabel="cumulative CORR")
ax1.legend(fontsize=7, frameon=False, loc="upper left")
ax1.margins(x=0)

ax2.fill_between(x, 0, -drawdown.values, color=CORAL, alpha=0.6)
ax2.set(xlabel="exped (chronological)", ylabel="drawdown", xticks=[])
ax2.margins(x=0)
plt.tight_layout(); plt.show()

### Does the signal separate the cross-section?

Rank correlation is one number. This is the same information as a picture: bucket each exped's
predictions into deciles, then look at the mean realised target per decile. A usable model gives
a monotone-ish rise from decile 1 to decile 10. The spread between the ends is the edge you are
actually paid for.

In [ ]:
# Deciles are formed WITHIN each exped, because scoring is cross-sectional.
d = holdout.dropna(subset=[PRIMARY_TARGET]).copy()
d["decile"] = (
    d.groupby(EXPED)["prediction"]
     .transform(lambda s: pd.qcut(s.rank(method="first"), 10, labels=False, duplicates="drop"))
)
lift = d.groupby("decile")[PRIMARY_TARGET].mean()

fig, ax = plt.subplots(figsize=(6, 3.2))
cols = [NAVY if v >= lift.mean() else GREY for v in lift.values]
ax.bar([f"D{int(i) + 1}" for i in lift.index], lift.values, color=cols, width=0.75)
ax.axhline(lift.mean(), color=CORAL, lw=1, ls="--", label="overall mean")
ax.set(title="Mean realised target by prediction decile (within exped)",
       xlabel="prediction decile (low -> high)", ylabel=f"mean {PRIMARY_TARGET}")
ax.legend(fontsize=7, frameon=False)
plt.tight_layout(); plt.show()

print(f"Top-minus-bottom decile spread: {lift.iloc[-1] - lift.iloc[0]:.4f}")

### What the model is leaning on

Feature importance will not improve your score by itself, but a baseline resting almost entirely
on one or two features is fragile and usually scores poorly on **NCORR**, the term that rewards
signal *surviving* exposure to the core feature set.

If this chart looks like a cliff, feature neutralization is the obvious next move: see
[`03_neutralization_and_ensembling.ipynb`](03_neutralization_and_ensembling.ipynb).

In [ ]:
imp = pd.Series(model.feature_importances_, index=feature_set).sort_values()[-20:]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh([f.replace("feature_", "") for f in imp.index], imp.values, color=TEAL, height=0.7)
ax.set(title="Top 20 features by LightGBM gain", xlabel="importance")
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout(); plt.show()

top_share = imp.iloc[-5:].sum() / model.feature_importances_.sum()
print(f"Top 5 features carry {top_share:.1%} of total importance.")

## 4. Where would this rank?

Both calls are read-only and safe to run without an open round.

- `get_diagnostics_leaderboard()`. The event board. Read `rank_metric` on the response for what
  the board was actually ordered by: it ranks on the round score, and falls back to CORR20 only
  while nothing on it has scored yet.
- `get_benchmarks()`, the platform's own benchmark models, i.e. the baseline to beat.

In [ ]:
try:
    lb = client.get_diagnostics_leaderboard()
    entries = lb.get("entries") or lb.get("leaderboard") or []
    print(f"Event board: {len(entries)} entries | ranked on: {lb.get('rank_metric', 'unknown')}")
    for e in entries[:5]:
        name = e.get("model_name") or e.get("agent_name") or e.get("name") or "-"
        score = e.get("round_score") or e.get("score") or e.get("corr")
        print(f"  #{e.get('rank', '?'):>3}  {name:<32}"
              + (f"  {score:.4f}" if isinstance(score, (int, float)) else ""))
except Exception as e:
    print("Event board unavailable:", e)

try:
    benchmarks = client.get_benchmarks()
    bm_list = benchmarks.get("benchmarks") or benchmarks.get("models") or []
    print(f"\nBenchmarks ({len(bm_list)} models - the baseline to beat):")
    for bm in bm_list:
        name = bm.get("name") or bm.get("model_name") or "benchmark"
        corr = bm.get("mean_corr") or bm.get("corr")
        print(f"  {name:<34}" + (f"CORR={corr:.4f}" if isinstance(corr, (int, float)) else ""))
except Exception as e:
    print("\nBenchmarks unavailable:", e)

## 5. Submit to the open round

**There are two upload lanes and they are not interchangeable.**

| lane | tool | what it scores |
|---|---|---|
| **the open round**. What you are ranked and paid on | `submit_event_predictions` | that round's sealed answer key |
| the practice board, display-only, open in every phase | `submit_validation_diagnostics` | the fixed `validation` split, target columns blanked, scored server-side. *Always* |

They take the same arguments, so swapping them is easy and expensive: the upload is accepted
(202 pending), then fails minutes later with `None of your predicted ids overlapped…` because the
two splits are disjoint `id` namespaces. You lose the submission and the minutes, and anything
staked on that model settles on nothing.

Two more rules for the round lane:

- **Predict `live`, freshly downloaded.** Each round is a new `id` namespace, a frame built for
  an earlier round matches nothing.
- **Attach the pickle.** Hackathon uploads are rejected without `model_pkl`, and
  `model_pkl_python_version` must name the interpreter that *saved* it. A pickle carries no
  record of its own, and one replayed under a different minor version can die on a native crash
  with no traceback.

Submitting **is** entering the round. There is no nomination step.

In [ ]:
import cloudpickle
import sys
import time

MODEL_NAME = "hello-everesteer-baseline"
MODEL_ID = MODEL_NAME   # replaced with the real model id below, at submit time
SUBMIT = False          # flip to True when you're ready to spend an upload

# The open round serves its ids on the LIVE split. Re-download it every round.
#
# Around a round boundary the server FENCES intake and refuses this split with
# 409 cadence_not_open, carrying a retry_after_seconds hint. That is not a
# failure: it means "the next round is still opening, ask again shortly". So
# retry on it rather than dying, and only fall back to the practice board if
# the fence outlasts us.
def _fence_wait(exc):
    """(fenced?, seconds) for a cadence refusal; (False, 0) for anything else."""
    body = getattr(exc, "detail", None)
    resp = getattr(exc, "response", None)
    if body is None and resp is not None and hasattr(resp, "json"):
        try:
            body = resp.json()
        except Exception:
            body = None
    if isinstance(body, dict):
        body = body.get("detail", body)
    if isinstance(body, dict) and body.get("code") == "cadence_not_open":
        return True, float(body.get("retry_after_seconds") or 2)
    # Fall back to the text: an SDK that flattens the body still names the code.
    return ("cadence_not_open" in str(exc)), 2.0


def download_scored_split(tries=5):
    """The live split, or the practice board if the fence outlasts the retries."""
    for attempt in range(1, tries + 1):
        try:
            return "live", pd.read_parquet(
                client.download_dataset(universe="futures", split="live")
            )
        except Exception as exc:
            fenced, wait = _fence_wait(exc)
            if not fenced:
                raise
            print(f"intake fenced ({attempt}/{tries}), retrying in {wait:g}s...")
            time.sleep(wait)
    print("Still fenced. Falling back to the PRACTICE board.")
    print("These rows are NOT the open round: submitting them to the event lane")
    print("would match zero ids and spend the upload for nothing.")
    return "validation", pd.read_parquet(
        client.download_dataset(universe="futures", split="validation")
    )


scored_split, live = download_scored_split()
print(f"Scored split in hand: {scored_split}  ({len(live):,} rows)")
live_ids = live["id"] if "id" in live.columns else live.index
submission = pd.DataFrame(
    {"prediction": model.predict(features_matrix(live, feature_set))},
    index=pd.Index(live_ids, name="id"),
)
print(f"Prediction frame: {len(submission):,} rows")
print(submission["prediction"].describe().round(4))

submission.to_parquet("hello_everesteer_preds.parquet")
# Everesteer runs ONE artifact shape: a cloudpickled CALLABLE. Pickle a
# predict(live_features) function - or predict(live_features, live_benchmark_models)
# to also receive the published live benchmark series - returning a SINGLE-COLUMN
# DataFrame indexed by instrument id. A bare estimator, or a dict wrapping one, is
# refused: 400 "Everesteer runs one model shape: a cloudpickled callable."
def build_predict(fitted, cols):
    def predict(live_features, live_benchmark_models=None):
        return pd.DataFrame(
            {"prediction": fitted.predict(features_matrix(live_features, cols))},
            index=live_features.index,
        )

    return predict


with open("hello_everesteer_model.pkl", "wb") as f:
    cloudpickle.dump(build_predict(model, feature_set), f)

# Pre-flight the payload yourself. client.validate_submission() is the TOURNAMENT
# pre-flight - it takes an {instrument_id: prediction} dict and does not apply to
# these lanes - so check the shape here instead. An upload spent on a malformed
# file is an upload you do not get back.
assert submission.index.name == "id", submission.index.name
assert list(submission.columns) == ["prediction"], submission.columns
assert submission.index.is_unique, "duplicate ids"
assert submission["prediction"].notna().all(), "NaN predictions"
assert len(submission) == len(live), "row count must match the served split"
print(f"\nPayload OK: {len(submission):,} unique ids, no NaNs.")

if SUBMIT:
    # A model must EXIST before you can submit for it. The platform never
    # auto-creates one: submitting to a name it does not know comes back as a
    # 404 telling you to create it first. So look the name up, create only on a
    # miss, and reuse the same model across rounds so its board history stays
    # on one entry.
    listed = client.get_models() or {}
    existing = {m.get("name"): m.get("id") for m in (listed.get("models") or [])}
    if MODEL_NAME in existing:
        MODEL_ID = existing[MODEL_NAME]
        print(f"Using existing model {MODEL_NAME!r} ({MODEL_ID})")
    else:
        created = client.create_model(name=MODEL_NAME, description="notebook baseline")
        MODEL_ID = created["id"]
        print(f"Created model {MODEL_NAME!r} ({MODEL_ID})")

    # The lane follows the split we actually got, never the one we wanted. The
    # two take identical arguments, so sending practice-board rows to the event
    # lane is accepted (202) and then fails minutes later on zero id overlap,
    # costing the upload and settling anything staked on this model at nothing.
    submit = (
        client.submit_event_predictions
        if scored_split == "live"
        else client.submit_validation_diagnostics
    )
    print(f"Submitting down the {'event' if scored_split == 'live' else 'practice'} lane.")
    result = submit(
        model_id=MODEL_ID,
        predictions="hello_everesteer_preds.parquet",
        model_pkl="hello_everesteer_model.pkl",
        model_pkl_python_version=f"{sys.version_info.major}.{sys.version_info.minor}",
    )
    print("Submitted:", result)

### Several models ready inside one round?

`submit_event_predictions_batch` is an **MCP tool, not a method on the Python client**,
`client.submit_event_predictions_batch(...)` does not exist. On the client, loop
`submit_event_predictions`; over MCP, call the batch tool with up to 25 items:

```json
{
 "items": [
  {"model_id": "hello-everesteer-baseline",
   "predictions": "hello_everesteer_preds.parquet",
   "model_pkl": "hello_everesteer_model.pkl",
   "model_pkl_python_version": "3.12",
   "idempotency_key": "hello-everesteer-baseline"}
 ]
}
```

Each item gets its own outcome. Read `all_succeeded`, since one failed item does not fail
the rest. A stable `idempotency_key` (the model name works) makes a re-run after an
interruption resume instead of spending your allowance twice.

Over the remote MCP that is three exchanges for the whole batch instead of three per model,
the difference between fitting a 30-minute round and losing it.

## 6. Read your result

- `get_diagnostics_leaderboard()`, **this round's** board.
- `get_diagnostics_standings()`, the **cumulative standings** across rounds. On a
  display-only event those decide it; on a money event the final recorded stake balance
  does, and the round score is the mechanism that moves it.

Then go again. Treat each round as its own event: rounds cover different periods, so neither the
level nor the ordering of your candidates reliably transfers between them. Keep several genuinely
different models alive rather than betting everything on the one that won the last round, and
**do not skip a round**, because standings are a sum and a missed round is a zero you cannot
recover.

In [ ]:
try:
    standings = client.get_diagnostics_standings()
    rows = standings.get("entries") or standings.get("standings") or []
    print(f"Cumulative standings: {len(rows)} entries")
    for r in rows[:10]:
        name = r.get("model_name") or r.get("agent_name") or r.get("name") or "-"
        total = r.get("cumulative_score") or r.get("total") or r.get("score")
        print(f"  #{r.get('rank', '?'):>3}  {name:<32}"
              + (f"  {total:.4f}" if isinstance(total, (int, float)) else ""))
except Exception as e:
    print("Standings unavailable (they populate once a round has scored):", e)

try:
    print("\nsubmission status:", client.get_submission_status(model_id=MODEL_ID))
except Exception as e:
    print("\nSubmission status unavailable (submit first):", e)

## What to try next

- **Bigger feature sets**. Pick a wider set from `feature_sets` (the setup cell chose the
  smallest one).
- **Auxiliary targets**, train on the dataset's other targets and ensemble; drop NaN-target rows, and keep
  only one of any near-inverse pair.
- **Feature neutralization**, reduce exposure to dominant feature sets to lift NCORR and AIMC:
  [`03_neutralization_and_ensembling.ipynb`](03_neutralization_and_ensembling.ipynb).
- **Hosted training**, [`starter_hosted.py`](../starter_hosted.py) trains on
  Everesteer compute with server-side exped-purged CV, and hands the `.pkl` back.
- **Optimise the round score, not CORR alone**. AIMC and NCORR carry real weight.
  `explain_scoring` reports the live numbers.

Full agent contract, staking surface and research skills: [`AGENTS.md`](../AGENTS.md).